# 1.1. Tiền xử lý dữ liệu


**Đọc dữ liệu, mô tả cấu trúc**



In [ ]:
# ==============================
# Imports & Global Settings
# ==============================
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (6,5), "axes.grid": True})

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, mean_absolute_error, mean_squared_error, accuracy_score
)
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

RND = 42
np.random.seed(RND)

DATA_PATH = "/content/typhoon_data_1970.csv"  # thay bằng đường dẫn đúng
HORIZONS = [6, 8, 12]  # giờ dự báo phía trước

# Tùy chọn hiển thị/đồ thị nhẹ
FAST_PLOTS = True


In [ ]:
# ==============================
# Mô tả cấu trúc và thống kê sơ bộ về dữ liệu
# ==============================

# Đọc dữ liệu thô
df = pd.read_csv(DATA_PATH)

print("📊 TỔNG QUAN DỮ LIỆU")
print(f"- Số bản ghi (rows): {df.shape[0]:,}")
print(f"- Số trường (columns): {df.shape[1]}")
print("\n🧩 DANH SÁCH CÁC TRƯỜNG:")
for col in df.columns:
    print(f"  • {col}")

print("\n📅 KIỂU DỮ LIỆU & GIÁ TRỊ KHÔNG RỖNG:")
df.info()

print("\n📈 THỐNG KÊ SƠ BỘ CÁC CỘT SỐ:")
display(df.describe().T[['count', 'mean', 'std', 'min', 'max']].round(2))

print("\n🔍 MỘT VÀI DÒNG DỮ LIỆU ĐẦU TIÊN:")
display(df.head())

# Kiểm tra giá trị thiếu
missing = df.isna().sum()
if missing.sum() > 0:
    print("\n⚠️ CÁC TRƯỜNG CÒN THIẾU GIÁ TRỊ:")
    display(missing[missing > 0].sort_values(ascending=False))
else:
    print("\n✅ KHÔNG CÓ GIÁ TRỊ THIẾU TRONG DỮ LIỆU.")


**Xử lý lỗi, thiếu, định dạng sai**

In [ ]:
# ==============================
#Load & Basic Cleaning  (FIX: normalize time tz → naive)
# ==============================
def memory_efficient_read_csv(path): #hàm đọc file
    df = pd.read_csv(path, encoding="utf-8", on_bad_lines="skip", low_memory=True) #on_bad_lines="skip": bỏ qua các dòng hỏng
    # Chuẩn hóa tên cột
    df.columns = df.columns.str.lower().str.strip()

    # Chuẩn hóa datetime
    if "iso_time" in df.columns:
        df["iso_time"] = pd.to_datetime(df["iso_time"], errors="coerce", utc=True) #chuyển toàn bộ giá trị thời gian từ t=string về datetime , errors="coerce" lỗi thì chuyển về NaT
        df = df.rename(columns={"iso_time": "time"}) #chuẩn hóa tên cột để thống nhất
    elif "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True)

#chuẩn hóa về cùng 1 time zone UTC giữa các châu lục và múi giờ khác nhau
    if "time" in df.columns:
        if hasattr(df["time"].dt, "tz"):
            # nếu tz-aware → đưa về UTC và bỏ tz
            try:
                df["time"] = df["time"].dt.tz_convert("UTC").dt.tz_localize(None)
            except Exception:
                # nếu không convert được (đã là UTC) thì chỉ bỏ tz
                df["time"] = df["time"].dt.tz_localize(None)

    # Chuẩn hóa tên các cột chính sid → storm_id ...
    df = df.rename(columns={
        "sid": "storm_id",
        "wmo_wind": "wind",
        "wmo_pres": "pres",
    })

    # Lọc những hàng có tọa độ hợp lệ
    df = df.dropna(subset=["lat","lon"]) # bỏ mọi hàng thiếu tọa độ

    # Sắp xếp theo storm_id, time (nếu có)
    key_time = "time" if "time" in df.columns else None # nếu df có cột time thì key_time sẽ nhận giá trị cột time
    if key_time:
        df = df.sort_values(["storm_id", key_time])# nếu có cột time thì sắp xếp theo tên id bão , sau đó sort theo thời gian
    else:
        df = df.sort_values(["storm_id"])
    df.reset_index(drop=True, inplace=True)
    return df #trả về dataframe đã chuẩn hóa và làm sạch

print("[1] Loading…")
df = memory_efficient_read_csv(DATA_PATH)
print(f"Shape: {df.shape} | time dtype: {df['time'].dtype if 'time' in df.columns else 'N/A'}")


In [ ]:
# ==============================
# Utilities (vectorized)
# ==============================
#CHÚ Ý : LAT:VĨ ĐỘ, LON; KINH ĐỘ
# Haversine vectorized: (lat1, lon1) -> (lat2, lon2), input dạng mảng numpy (deg)
def haversine_vec(lat1, lon1, lat2, lon2): #hàm tính khoảng cách địa lí giữa 2 điểm
    # Chuyển sang radian vì hàm sin cos của numpy tính bằng radian
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1 #tính chênh lệch kinh độ và vĩ độ
    dlon = lon2 - lon1
    # công thức chuẩn của Haversine:
    # a đo độ cong giữa hai điểm.
    # c là góc tâm giữa hai điểm trên quả cầu.
    # R là bán kính Trái Đất ≈ 6371 km.
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*(np.sin(dlon/2.0)**2)
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    R = 6371.0
    return R * c  # R * c cho ra khoảng cách thực tế trên bề mặt Trái Đất (km).

def group_split_indices(X, y, groups, test_size): # chia dữ liệu train/test mà vẫn giữ toàn vẹn từng cơn bão (storm_id).
    splitter = GroupShuffleSplit(test_size=test_size, n_splits=1, random_state=RND) #GroupShuffleSplit là một công cụ trong sklearn.model_selection cho phép chia dữ liệu theo nhóm
    return next(splitter.split(X, y, groups)) # X, y: dữ liệu và nhãn
    # groups: cột định danh nhóm (ở đây là storm_id),
    # test_size: tỉ lệ chia test (ví dụ 0.3).
    # Hàm .split() sẽ chọn ngẫu nhiên một số nhóm (storm_id) cho tập test, còn lại cho train.
    # next() chỉ lấy ra lần chia đầu tiên (vì n_splits=1).


In [ ]:
# ==============================
# 3) Feature Engineering (leak-free)  (FIX: robust datetime dtype check)
# ==============================
# FEATURING: để biến biến dữ liệu thô thành dữ liệu mô hình có thể học được.
# Mọi đặc trưng chỉ dùng thông tin hiện tại & quá khứ (lag/rolling) — không dùng tương lai.

def add_time_features(d): #thêm đặc trung cho dataframe
    d = d.copy()
    # Dùng checker của pandas (hỗ trợ cả tz-aware/naive)
    if "time" in d.columns and pd.api.types.is_datetime64_any_dtype(d["time"]):
        # đảm bảo vẫn là naive (nếu Cell 2 chưa strip tz vì lý do nào đó)
        try:
            if hasattr(d["time"].dt, "tz") and d["time"].dt.tz is not None:
                d["time"] = d["time"].dt.tz_localize(None) #bỏ timezone (múi giờ) , chuẩn hóa hết về UTC
        except Exception:
            pass
        d["month"] = d["time"].dt.month.astype("Int16") # tạo cột month hour dayofyear cho model học, astype("Int16") tiết kiệm bộ nhớ so với int64
        d["hour"] = d["time"].dt.hour.astype("Int16")
        d["dayofyear"] = d["time"].dt.dayofyear.astype("Int16")
    else:
        d["month"] = pd.Series(pd.NA, index=d.index, dtype="Int16") # Nếu dữ liệu không có time, tạo cột rỗng (pd.NA) để tránh lỗi downstream.
        d["hour"] = pd.Series(pd.NA, index=d.index, dtype="Int16")
        d["dayofyear"] = pd.Series(pd.NA, index=d.index, dtype="Int16")
    return d

def add_lag_kinematics(d, max_lag_steps=2):
# Thêm đặc trưng động học (kinematic features) – tức là mô tả “chuyển động” của bão dựa vào dữ liệu quá khứ , max_lag_steps=2 nghĩa là tạo đặc trưng lag cho 2 bước trước đó
#(ví dụ: nếu dữ liệu ghi mỗi 3h → ta sẽ có 3h trước và 6h trước).
#chỉ lấy 2 mốc thời gian trước để dự đoán cho tương lai vì tìn hiệu ở quá khứ quá xa nó gây nhiễu, với cả dễ overfit nếu dữ liệu không đủ số bản ghi
    d = d.copy()
    d = d.sort_values(["storm_id","time"]).reset_index(drop=True) # sắp xếp bão theo id và thời gian
    # LAGs theo quá khứ
    for L in range(1, max_lag_steps+1):
        d[f"lat_lag{L}"] = d.groupby("storm_id")["lat"].shift(L) #dịch giá trị xuống L hàng để lấy vĩ độ ở mốc thời gian trước
        d[f"lon_lag{L}"] = d.groupby("storm_id")["lon"].shift(L) #dịch giá trị xuống L hàng để lấy kinh độ ở mốc thời gian trước
        d[f"dlat_lag{L}"] = d["lat"] - d[f"lat_lag{L}"] # tính hiệu di chuyển
        d[f"dlon_lag{L}"] = d["lon"] - d[f"lon_lag{L}"]

    # Khoảng cách (km) so với bước trước (lag1) — vector hóa
    lat_lag1 = d["lat_lag1"].to_numpy()
    lon_lag1 = d["lon_lag1"].to_numpy()
    lat_cur  = d["lat"].to_numpy()
    lon_cur  = d["lon"].to_numpy()
    step_km = np.full(len(d), np.nan, dtype=float)
    mask = (~pd.isna(lat_lag1)) & (~pd.isna(lon_lag1))
    step_km[mask] = haversine_vec(lat_lag1[mask], lon_lag1[mask], lat_cur[mask], lon_cur[mask]) #tính khoảng cách mà bão đã di chuyển trong 3h gần nhất dùng hàm haversine_vec
    d["step_km_lag1"] = step_km #thêm đặc trưng mới

    # Rolling trung bình 3 bước gần nhất (quá khứ)
    d["step_km_roll3"] = (
        d.groupby("storm_id")["step_km_lag1"]
         .transform(lambda x: x.rolling(window=3, min_periods=1).mean()) #tính vẫn tốc trung bình trong 3 bước , tức 9 tiếng gần nhất
    )
    return d

# === ORIGINAL HORIZON: dịch theo số bước 3h ===
#horizon :khoảng thời gian trong tương lai muốn dự báo
# hàm này nói cho mô hình biết “điều cần dự đoán là gì”.
def build_horizon(df_in, horizon_hours=6, base_step_hours=3):
    steps = max(1, int(round(horizon_hours / base_step_hours))) #tính số bước, lấy horizon chia base step (ví dụ basestep 3 tiếng mà muốn dự đoán 9 tiếng sau thì 3 bước)
    d = df_in.copy() #sao chép dataframe
    d = d.sort_values(["storm_id", "time"]).reset_index(drop=True)
    d["lat_tgt"] = d.groupby("storm_id")["lat"].shift(-steps) # groupby("storm_id") bảo đảm shift chỉ trong cùng một bão
    d["lon_tgt"] = d.groupby("storm_id")["lon"].shift(-steps) # shift(-steps) dịch lên (nghĩa là lấy giá trị ở tương lai so với hàng hiện tại).
    d = d.dropna(subset=["lat_tgt","lon_tgt"]).reset_index(drop=True)
    return d #trả về dataframe dùng làm target



# Áp dụng
df_feat = add_time_features(df)
df_feat = add_lag_kinematics(df_feat, max_lag_steps=2)
print("[3] Features added (leak-free). Columns:", len(df_feat.columns))


In [ ]:
# ==============================
# 4) Build Horizon Datasets
# ==============================
#tạo nhiều dataset tương ứng với nhiều chiều thời gian tương lai (6-8-12)
dfs = {}
for h in HORIZONS: #HORIZONS khai báo ở cell đầu
    D = build_horizon(df_feat, horizon_hours=h, base_step_hours=3) # D là một bản sao của df_feat nhưng có thêm cột mục tiêu tương lai (lat_tgt, lon_tgt) cho mốc h
    dfs[h] = D
    print(f"Horizon {h}h → {len(D):,} rows")



#1.2. Phân tích và trực quan hóa dữ liệu:

In [ ]:
# ==============================
# 8) Trực quan hóa mối quan hệ giữa đặc trưng chính và đầu ra
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

H = 6
D = dfs[H].sample(30000, random_state=42).copy()

# Nếu có hàm add_dynamic_features thì thêm đặc trưng
try:
    D = add_dynamic_features(D)
except NameError:
    pass

num_cols = [c for c in D.select_dtypes("number").columns if c not in ("lat_tgt", "lon_tgt")]

# --- Tính tương quan Pearson ---
corr_lat = D[num_cols + ["lat_tgt"]].corr()["lat_tgt"].drop("lat_tgt").sort_values(ascending=False)
corr_lon = D[num_cols + ["lon_tgt"]].corr()["lon_tgt"].drop("lon_tgt").sort_values(ascending=False)

# --- Top đặc trưng tương quan cao nhất ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=corr_lat.head(10), y=corr_lat.head(10).index, ax=axes[0], color="skyblue")
axes[0].set_title("🔹 Tương quan với vĩ độ dự đoán (lat_tgt)")
sns.barplot(x=corr_lon.head(10), y=corr_lon.head(10).index, ax=axes[1], color="lightgreen")
axes[1].set_title("🔸 Tương quan với kinh độ dự đoán (lon_tgt)")
plt.tight_layout()
plt.show()

# --- Scatterplot minh họa (lấy đặc trưng tương quan mạnh nhất) ---
feat_lat = corr_lat.index[0]
feat_lon = corr_lon.index[0]

plt.figure(figsize=(6,5))
sns.scatterplot(x=D[feat_lat], y=D["lat_tgt"], alpha=0.4, color="blue")
plt.title(f"Quan hệ tuyến tính giữa {feat_lat} và lat_tgt")
plt.show()

plt.figure(figsize=(6,5))
sns.scatterplot(x=D[feat_lon], y=D["lon_tgt"], alpha=0.4, color="green")
plt.title(f"Quan hệ tuyến tính giữa {feat_lon} và lon_tgt")
plt.show()


In [ ]:
# ==============================
# 9) Trực quan hóa dữ liệu theo các cặp thành phần chính (PCA visualization)
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

H = 6  # ví dụ horizon 6h
D = dfs[H].sample(30000, random_state=42).copy()

# Lấy cột numeric để giảm chiều
num_cols = [c for c in D.select_dtypes("number").columns if c not in ("lat_tgt", "lon_tgt")]
X = D[num_cols].fillna(D[num_cols].median())
X_scaled = StandardScaler().fit_transform(X)

# PCA giữ 6 thành phần chính
pca = PCA(n_components=6, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# DataFrame cho trực quan
pca_df = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(6)])
pca_df["lat_tgt"] = D["lat_tgt"]
pca_df["lon_tgt"] = D["lon_tgt"]

# Hiển thị % phương sai giải thích
explained = np.round(pca.explained_variance_ratio_ * 100, 2)
print("📊 Phương sai giải thích của 6 thành phần đầu tiên:", explained)
print("Tổng cộng:", explained.sum().round(2), "%")

# Vẽ scatter cho các cặp thành phần chính
pairs = [(0, 1), (1, 2), (2, 3)]
plt.figure(figsize=(15, 10))
for i, (a, b) in enumerate(pairs, 1):
    plt.subplot(2, 2, i)
    sns.scatterplot(x=pca_df[f"PC{a+1}"], y=pca_df[f"PC{b+1}"],
                    hue=pca_df["lat_tgt"], palette="viridis", alpha=0.5, legend=False)
    plt.title(f"PC{a+1} vs PC{b+1}")
plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# Đánh giá PCA & LDA theo số chiều
# ==============================

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
import numpy as np

# Data
D = dfs[HORIZONS[0]]
X = D.select_dtypes("number").drop(columns=["lat_tgt","lon_tgt"], errors="ignore")

Xn = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler())
]).fit_transform(X)

labels = KMeans(n_clusters=4, random_state=42, n_init="auto").fit_predict(Xn)

print("===== PCA: Explained variance =====")
for k in [2, 4, 6, 8, 12]:
    pca = PCA(n_components=k, random_state=42)
    pca.fit(Xn)
    print(f"k={k:2d} → {pca.explained_variance_ratio_.sum():.2%}")

print("\n===== LDA: Fisher score =====")
for k in [1, 2, 3]:  # max = C-1 = 3
    lda = LDA(n_components=k)
    Xl = lda.fit_transform(Xn, labels)
    centroids = np.vstack([Xl[labels==c].mean(0) for c in np.unique(labels)])
    fisher = pairwise_distances(centroids).mean() / np.mean(
        [np.var(Xl[labels==c], axis=0).sum() for c in np.unique(labels)]
    )
    print(f"k={k} → Fisher = {fisher:.4f}")


In [ ]:
# ==============================
# Phân bố dữ liệu theo LDA (1D)
# ==============================

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# LDA 1 chiều (tối ưu phân biệt)
lda_1d = LDA(n_components=1)
Xl_1d = lda_1d.fit_transform(Xn, labels)

# Plot histogram
plt.figure(figsize=(8,5))
for c in np.unique(labels):
    sns.histplot(
        Xl_1d[labels == c, 0],
        bins=30,
        stat="count",
        alpha=0.6,
        label=f"Class {c}"
    )

plt.xlabel("LDA Component 1")
plt.ylabel("Tần suất")
plt.title("Phân bố dữ liệu theo thành phần LDA")
plt.legend()
plt.tight_layout()
plt.show()


### Đánh giá PCA và LDA theo số chiều

Kết quả PCA cho thấy khi chỉ sử dụng 2 thành phần chính, dữ liệu chỉ giữ lại khoảng 50% phương sai, chưa đủ để bảo toàn thông tin. Khi tăng số chiều, lượng phương sai giữ lại tăng nhanh; với 6–8 thành phần chính đã giữ trên 84–92% phương sai, cho thấy dữ liệu có thể được nén hiệu quả ở không gian thấp chiều nhưng không phù hợp nếu cố định ở 2 chiều.

Đối với LDA, Fisher score đạt giá trị cao nhất ở 1 chiều và giảm dần khi tăng số chiều. Điều này cho thấy hướng phân biệt mạnh nhất nằm chủ yếu trên trục phân biệt đầu tiên, trong khi các trục tiếp theo đóng góp ít hơn và có xu hướng làm giảm tỷ lệ giữa độ phân tán giữa các lớp và trong từng lớp. Kết quả này phản ánh rằng cấu trúc phân biệt của dữ liệu mang tính tuyến tính và tập trung chủ yếu theo một hướng chính.


In [ ]:
# ==============================
# Mô tả dữ liệu sau khi chuẩn hóa
# ==============================
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

example_h = HORIZONS[0]
D = dfs[example_h].copy()

# --- Chọn các cột số, loại bỏ đầu ra ---
num_cols = [c for c in D.select_dtypes("number").columns if c not in ("lat_tgt", "lon_tgt")]

# --- Tiền xử lý như pipeline đã dùng ---
imp = SimpleImputer(strategy="median")
sc = StandardScaler()

X_imp = pd.DataFrame(imp.fit_transform(D[num_cols]), columns=num_cols)
X_scaled = pd.DataFrame(sc.fit_transform(X_imp), columns=num_cols)

# --- Thống kê mô tả ---
desc = X_scaled.describe().T[['mean','std','min','max']].round(2)
desc = desc.rename(columns={'mean':'Giá trị trung bình','std':'Độ lệch chuẩn','min':'Nhỏ nhất','max':'Lớn nhất'})
display(desc.style.background_gradient(cmap="YlGnBu").set_caption("📊 Thống kê dữ liệu sau chuẩn hóa (StandardScaler)"))

print(f"Tổng số đặc trưng sau chuẩn hóa: {X_scaled.shape[1]}")
print(f"Số bản ghi: {X_scaled.shape[0]}")

# --- Nhận xét nhanh ---
print("\n🧩 Nhận xét:")
print("- Tất cả các cột đều có mean ≈ 0 và std ≈ 1 → Chuẩn hóa thành công.")
print("- Các giá trị min/max thể hiện mức độ phân tán sau chuẩn hóa, không còn lệch lớn giữa các đặc trưng.")
print("- Bộ dữ liệu hiện sẵn sàng cho các bước giảm chiều (PCA) hoặc huấn luyện mô hình hồi quy.")


Phân tích kết quả chuẩn hóa
🔹 1️⃣ Ý nghĩa của bảng thống kê

Sau khi chuẩn hóa bằng StandardScaler, tất cả các đặc trưng số đều có:

Giá trị trung bình (mean) ≈ 0

Độ lệch chuẩn (std) ≈ 1

Điều này xác nhận rằng dữ liệu đã được đưa về cùng một thang đo, tức là mọi biến đều có mức ảnh hưởng tương đương khi đi vào mô hình — đặc biệt quan trọng cho các thuật toán như PCA, Ridge, hay MLP.

🔹 2️⃣ Mức độ phân tán (min, max)

Các giá trị min/max nằm trong khoảng từ -8 đến 10, nghĩa là vẫn có một số giá trị ngoại lệ (outlier), nhưng không còn sự chênh lệch cực lớn giữa các cột như trước khi chuẩn hóa (ví dụ trước đó dist2land có thể hàng nghìn km trong khi lat chỉ vài chục độ).

Điều này giúp:

PCA giảm chiều ổn định hơn, không bị chi phối bởi biến lớn.

MLP hội tụ nhanh hơn, vì gradient descent không bị lệch hướng.

🔹 3️⃣ Kết luận mô tả

Bộ dữ liệu sau chuẩn hóa có 19 đặc trưng số, với 127.690 bản ghi hợp lệ.
Dữ liệu đã cân bằng về thang đo, sẵn sàng cho các bước giảm chiều (PCA) hoặc huấn luyện mô hình hồi quy.
Việc chuẩn hóa giúp tránh hiện tượng “biến lớn át biến nhỏ”, cải thiện độ ổn định của quá trình huấn luyện.

In [ ]:
# ==============================
# Feature Engineering mở rộng: thêm đặc trưng động học
# ==============================

def add_dynamic_features(D):
    D = D.copy()
    # Tốc độ di chuyển trung bình (km/h) – mỗi bước 3h
    D["speed_kmh"] = D["step_km_lag1"] / 3.0

    # Hướng di chuyển (bearing, độ)
    D["bearing"] = np.degrees(np.arctan2(D["dlon_lag1"], D["dlat_lag1"]))
    D["bearing"] = D["bearing"].fillna(0)

    # Gia tốc theo thời gian
    D["dlat_acc"] = D["dlat_lag1"] - D["dlat_lag2"]
    D["dlon_acc"] = D["dlon_lag1"] - D["dlon_lag2"]

    # Khoảng cách tới xích đạo (độ)
    D["dist2eq"] = np.abs(D["lat"])

    return D

# Áp dụng cho toàn bộ horizons
dfs = {h: add_dynamic_features(df) for h, df in dfs.items()}
print(" Đã thêm đặc trưng động học: speed_kmh, bearing, dlat_acc, dlon_acc, dist2eq")


#1.4. Phân tích hồi quy

In [ ]:
# ==============================
# Regression Optimized (RAM/Speed/Accuracy)
# RidgeCV (sparse, no PCA) + MLP (SVD-128) + Feature Expansion
# ==============================
import numpy as np, pandas as pd, gc, warnings
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RND = globals().get("RND", 42)
np.random.seed(RND)

# ---- Optional: lấy mẫu để test nhanh. Đặt None để dùng full dữ liệu.
SAMPLE_SIZE = None  # ví dụ 30000 để test nhanh; None để train thật

# ---- Haversine (km)
def haversine_vec(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))

def geodesic_km_avg(y_true, y_pred):
    return float(np.nanmean(haversine_vec(
        y_true[:,0], y_true[:,1], y_pred[:,0], y_pred[:,1]
    )))

# ---- Feature engineering mở rộng
def add_dynamic_features(D):
    D = D.copy()
    if "step_km_lag1" in D:
        D["speed_kmh"] = D["step_km_lag1"] / 3.0
    else:
        D["speed_kmh"] = 0.0
    D["bearing"] = np.degrees(np.arctan2(D.get("dlon_lag1", 0.0), D.get("dlat_lag1", 0.0)))
    D["bearing"] = D["bearing"].fillna(0)
    D["dlat_acc"] = D.get("dlat_lag1", 0.0) - D.get("dlat_lag2", 0.0)
    D["dlon_acc"] = D.get("dlon_lag1", 0.0) - D.get("dlon_lag2", 0.0)
    D["dist2eq"] = np.abs(D.get("lat", 0.0))
    return D

# ---- Chuẩn hóa categorical an toàn
def _sanitize_categoricals(df_in, cat_cols):
    df = df_in.copy()
    for c in cat_cols:
        df[c] = df[c].astype(str)
        df[c] = df[c].replace("nan", "__MISSING__")
    return df

# ---- OHE helper (sparse cho RAM thấp)
def make_ohe_sparse():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True,
                             min_frequency=0.005, max_categories=64)
    except TypeError:
        # fallback cho sklearn cũ
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def regression_optimized(D, test_size=0.3, svd_components=128):
    # (1) Lấy mẫu nhẹ nếu cần
    if SAMPLE_SIZE is not None and len(D) > SAMPLE_SIZE:
        D = D.sample(SAMPLE_SIZE, random_state=RND).reset_index(drop=True)

    # (2) Feature expansion
    D = add_dynamic_features(D)

    # (3) Chọn cột
    cat_cols = D.select_dtypes("object").columns.tolist()
    num_cols = [c for c in D.select_dtypes("number").columns if c not in ("lat_tgt", "lon_tgt")]

    # (4) X/y/groups
    D_cat = _sanitize_categoricals(D[cat_cols], cat_cols) if cat_cols else pd.DataFrame(index=D.index)
    X_all = pd.concat([D_cat, D[num_cols]], axis=1) if cat_cols else D[num_cols]
    y_all = D[["lat_tgt", "lon_tgt"]].values
    if "storm_id" in D.columns:
        groups = D["storm_id"].astype(str).values
    else:
        groups = np.arange(len(D))  # fallback

    # (5) Chia train/test theo nhóm bão
    gss = GroupShuffleSplit(test_size=test_size, n_splits=1, random_state=RND)
    tr_idx, te_idx = next(gss.split(X_all, y_all, groups))
    Xtr, Xte = X_all.iloc[tr_idx], X_all.iloc[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]

    # ---------------------------
    # RIDGE: num (impute+scale), cat (OHE sparse), NO PCA  → chính xác & nhanh
    # ---------------------------
    num_pipe_ridge = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())  # giữ dense cho ít cột num, OK
    ])
    pre_ridge = ColumnTransformer([
        ("num", num_pipe_ridge, num_cols),
        ("cat", make_ohe_sparse(), cat_cols)
    ], remainder="drop", sparse_threshold=1.0)  # ép ưu tiên sparse nếu có

    ridge = MultiOutputRegressor(
        RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0], cv=3),
        n_jobs=None  # tránh nested parallelism chiếm RAM
    )
    pipe_ridge = Pipeline([("pre", pre_ridge), ("est", ridge)])
    pipe_ridge.fit(Xtr, ytr)
    pr = pipe_ridge.predict(Xte)

    ridge_res = {
        "model": "RidgeCV (no PCA, sparse OHE)",
        "rmse": float(np.sqrt(mean_squared_error(yte, pr))),
        "mae": float(mean_absolute_error(yte, pr)),
        "geo_km": geodesic_km_avg(yte, pr)
    }

    # ---------------------------
    # MLP: num+cat (cat sparse) → SVD(128) → Scale → MLP(early stopping)
    # ---------------------------
    num_pipe_mlp = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ])
    pre_mlp_sparse = ColumnTransformer([
        ("num", num_pipe_mlp, num_cols),
        ("cat", make_ohe_sparse(), cat_cols)
    ], remainder="drop", sparse_threshold=1.0)

    mlp = MLPRegressor(
        hidden_layer_sizes=(128, 64),
        solver="adam",
        alpha=1e-4,
        batch_size=256,         # RAM-friendly
        learning_rate_init=1e-3,
        max_iter=400,
        random_state=RND,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12
    )

    # SVD giảm còn dense nhỏ gọn; scale lại để MLP học ổn định
    mlp_pipeline = Pipeline([
        ("pre", pre_mlp_sparse),                         # sparse (rất lớn) ->
        ("svd", TruncatedSVD(n_components=svd_components, random_state=RND)),  # -> dense nhỏ
        ("sc2", StandardScaler()),                       # scale trên không gian nén
        ("est", MultiOutputRegressor(mlp))
    ])

    mlp_pipeline.fit(Xtr, ytr)
    pm = mlp_pipeline.predict(Xte)

    # lưu số vòng học thực tế
    try:
        epoch_run = mlp_pipeline.named_steps["est"].estimator_.n_iter_
    except Exception:
        epoch_run = None

    svd_var = float(np.sum(mlp_pipeline.named_steps["svd"].explained_variance_ratio_))

    mlp_res = {
        "model": f"MLP (SVD-{svd_components}, early stop)",
        "rmse": float(np.sqrt(mean_squared_error(yte, pm))),
        "mae": float(mean_absolute_error(yte, pm)),
        "geo_km": geodesic_km_avg(yte, pm),
        "svd_var_kept": round(svd_var, 4),
        "epoch_run": int(epoch_run) if epoch_run is not None else None
    }

    # dọn RAM
    del Xtr, Xte, ytr, yte
    gc.collect()

    return pd.DataFrame([ridge_res, mlp_res])

# ======= CHẠY CHO TỪNG HORIZON =======
from IPython.display import display
results_opt = {}
for h in HORIZONS:
    print(f"\n[Optimized Regression] Horizon {h}h  (sample={SAMPLE_SIZE})")
    D = dfs[h].copy()
    results_opt[h] = regression_optimized(D, test_size=0.3, svd_components=128)
    display(results_opt[h])


So sánh kết quả của các tỉ lệ chia train test

In [ ]:
# ==============================
# So sánh tỷ lệ train/test — bản 30k dòng (nhanh & ổn định)
# ==============================
SAMPLE_SIZE = 30000     # cố định 30.000 dòng
ratios = [0.2, 0.3, 0.4]  # 4:1, 7:3, 6:4

from IPython.display import display

results_opt_ratios = {}
for h in [6]:  # chỉ test horizon 6h để tăng tốc
    print(f"\n📊 [Quick Ratio Test] Horizon {h}h (sample={SAMPLE_SIZE} rows)")
    D = dfs[h].copy()

    # ✅ chọn ngẫu nhiên 30k dòng (nếu dữ liệu nhiều hơn)
    if len(D) > SAMPLE_SIZE:
        D = D.sample(n=SAMPLE_SIZE, random_state=RND).reset_index(drop=True)

    rows = []
    for ts in ratios:
        print(f"  ▶️ Test_size={ts}  (train={1-ts:.1f})")
        res = regression_optimized(D, test_size=ts, svd_components=64)  # nhẹ hơn để chạy nhanh
        res["test_size"] = ts
        rows.append(res)

    results_opt_ratios[h] = pd.concat(rows, ignore_index=True)
    display(results_opt_ratios[h])


Biểu đồ so sánh theo tỉ lệ train test


In [ ]:
# ==============================
# Vẽ biểu đồ so sánh RMSE / MAE / GeoKM theo tỷ lệ train-test
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

# ⚠️ Dán bảng của bạn vào đây nếu chưa có sẵn:
data = pd.DataFrame({
    "model": [
        "RidgeCV (no PCA, sparse OHE)", "MLP (SVD-64, early stop)",
        "RidgeCV (no PCA, sparse OHE)", "MLP (SVD-64, early stop)",
        "RidgeCV (no PCA, sparse OHE)", "MLP (SVD-64, early stop)"
    ],
    "rmse": [0.307039, 3.098335, 0.304784, 3.127594, 0.309829, 3.381714],
    "mae": [0.198438, 1.944549, 0.200051, 2.023845, 0.203331, 2.204377],
    "geo_km": [32.699497, 356.523987, 32.913463, 362.509625, 33.422291, 391.054470],
    "test_size": [0.2, 0.2, 0.3, 0.3, 0.4, 0.4]
})

# --- Vẽ ---
metrics = ["rmse", "mae", "geo_km"]
titles = ["Sai số RMSE", "Sai số MAE", "Khoảng cách địa lý trung bình (km)"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, m in enumerate(metrics):
    sns.barplot(
        data=data, x="test_size", y=m, hue="model",
        palette="Set2", ax=axes[i]
    )
    axes[i].set_title(titles[i])
    axes[i].set_xlabel("Tỷ lệ test (test_size)")
    axes[i].set_ylabel(m.upper())
    axes[i].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


Thực hiện hồi quy với dữ liệu giảm chiều

In [ ]:
# ==============================
#  Compare RidgeCV / MLP — Original vs PCA (sample=30k, optimized)
# ==============================
import numpy as np, pandas as pd, gc, warnings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from IPython.display import display

warnings.filterwarnings("ignore")
RND = 42
SAMPLE_SIZE = 30000

# ---- Feature expansion
def add_dynamic_features(D):
    D = D.copy()
    if "step_km_lag1" in D:
        D["speed_kmh"] = D["step_km_lag1"] / 3.0
    else:
        D["speed_kmh"] = 0.0
    D["bearing"] = np.degrees(np.arctan2(D.get("dlon_lag1", 0.0), D.get("dlat_lag1", 0.0)))
    D["bearing"] = D["bearing"].fillna(0)
    D["dlat_acc"] = D.get("dlat_lag1", 0.0) - D.get("dlat_lag2", 0.0)
    D["dlon_acc"] = D.get("dlon_lag1", 0.0) - D.get("dlon_lag2", 0.0)
    D["dist2eq"] = np.abs(D.get("lat", 0.0))
    return D

# ---- Safe OHE
def make_ohe_sparse():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True,
                             min_frequency=0.005, max_categories=64)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

# ---- Đo khoảng cách địa lý
def geodesic_km_avg(y_true, y_pred):
    lat1, lon1, lat2, lon2 = map(np.radians, [y_true[:,0], y_true[:,1], y_pred[:,0], y_pred[:,1]])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a)).mean()

# ---- Hàm so sánh
def regression_compare_opt(D, use_pca=False, n_components=0.9, test_size=0.3):
    # 1️⃣ Lấy mẫu nhỏ để chạy nhanh
    D = D.sample(n=min(SAMPLE_SIZE, len(D)), random_state=RND).reset_index(drop=True)
    D = add_dynamic_features(D)

    # 2️⃣ Chia cột
    cat_cols = D.select_dtypes("object").columns.tolist()
    num_cols = [c for c in D.select_dtypes("number").columns if c not in ("lat_tgt","lon_tgt")]

    for c in cat_cols:
        D[c] = D[c].astype(str).fillna("__MISSING__")

    y = D[["lat_tgt","lon_tgt"]].values
    groups = D["storm_id"].astype(str).values

    # 3️⃣ Chia train/test theo bão
    gss = GroupShuffleSplit(test_size=test_size, n_splits=1, random_state=RND)
    tr_idx, te_idx = next(gss.split(D, y, groups))
    Xtr, Xte, ytr, yte = D.iloc[tr_idx], D.iloc[te_idx], y[tr_idx], y[te_idx]

    # 4️⃣ Pipeline numeric
    num_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ])
    if use_pca:
        num_pipe.steps.append(("pca", PCA(n_components=n_components, random_state=RND)))

    # 5️⃣ Preprocessor chung (sparse)
    preproc = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", make_ohe_sparse(), cat_cols)
    ], remainder="drop", sparse_threshold=1.0)

    # 6️⃣ RidgeCV
    ridge = MultiOutputRegressor(RidgeCV(alphas=[0.1,1.0,10.0,50.0], cv=3), n_jobs=None)
    pipe_ridge = Pipeline([("pre", preproc), ("est", ridge)])
    pipe_ridge.fit(Xtr, ytr)
    pr = pipe_ridge.predict(Xte)

    # 7️⃣ MLP (tinh chỉnh nhẹ)
    mlp = MLPRegressor(hidden_layer_sizes=(128,64),
                       alpha=1e-3,
                       batch_size=512,
                       learning_rate_init=1e-3,
                       max_iter=400,
                       early_stopping=True,
                       validation_fraction=0.15,
                       n_iter_no_change=12,
                       random_state=RND)
    pipe_mlp = Pipeline([("pre", preproc), ("est", MultiOutputRegressor(mlp))])
    pipe_mlp.fit(Xtr, ytr)
    pm = pipe_mlp.predict(Xte)

    return pd.DataFrame([
        {"model": f"RidgeCV {'+ PCA' if use_pca else '(original)'}",
         "rmse": np.sqrt(mean_squared_error(yte, pr)),
         "mae": mean_absolute_error(yte, pr),
         "geo_km": geodesic_km_avg(yte, pr)},
        {"model": f"MLP {'+ PCA' if use_pca else '(original)'}",
         "rmse": np.sqrt(mean_squared_error(yte, pm)),
         "mae": mean_absolute_error(yte, pm),
         "geo_km": geodesic_km_avg(yte, pm)}
    ])

# ==============================
# 🔹 Chạy so sánh (ví dụ horizon 6h)
# ==============================
H = 6
D = dfs[H].copy()
print(f"\n📊 [Compare RidgeCV/MLP Optimized] Horizon {H}h, sample={SAMPLE_SIZE}")

res_orig = regression_compare_opt(D, use_pca=False)
res_pca  = regression_compare_opt(D, use_pca=True)
compare_opt = pd.concat([res_orig, res_pca], ignore_index=True)
display(compare_opt)


Kiểm tra Overfitting

In [ ]:
# ==============================
# Kiểm tra Overfitting — RidgeCV vs MLP (Pipeline tối ưu, fix encoder bug)
# ==============================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, gc
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import RidgeCV
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor

# --- Cấu hình
H = 6
SAMPLE_SIZE = 30000
np.random.seed(42)

# --- Lấy mẫu dữ liệu nhỏ để test nhanh
D = dfs[H].sample(SAMPLE_SIZE, random_state=42).copy()
D = add_dynamic_features(D)

# --- 🔧 Ép tất cả categorical về chuỗi, tránh lỗi float/str lẫn lộn
for c in D.columns:
    if D[c].dtype == "object" or D[c].nunique() < 25:
        D[c] = D[c].astype(str).fillna("__MISSING__")

# --- Chia nhóm train/test theo bão
y_all = D[["lat_tgt", "lon_tgt"]].values
groups = D["storm_id"].astype(str).values
gss = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
tr_idx, te_idx = next(gss.split(D, y_all, groups))
Xtr, Xte = D.iloc[tr_idx], D.iloc[te_idx]
ytr, yte = y_all[tr_idx], y_all[te_idx]

# --- Chuẩn bị cột
cat_cols = Xtr.select_dtypes("object").columns.tolist()
num_cols = [c for c in Xtr.select_dtypes("number").columns if c not in ("lat_tgt","lon_tgt")]

# --- RIDGECV (pipeline tối ưu)
num_pipe_ridge = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler())
])
pre_ridge = ColumnTransformer([
    ("num", num_pipe_ridge, num_cols),
    ("cat", make_ohe_sparse(), cat_cols)
], sparse_threshold=1.0)

ridge_model = MultiOutputRegressor(
    RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0], cv=3),
    n_jobs=None
)
ridge_pipe = Pipeline([("pre", pre_ridge), ("est", ridge_model)])
ridge_pipe.fit(Xtr, ytr)

pred_train_ridge = ridge_pipe.predict(Xtr)
pred_test_ridge  = ridge_pipe.predict(Xte)
ridge_train_rmse = np.sqrt(mean_squared_error(ytr, pred_train_ridge))
ridge_test_rmse  = np.sqrt(mean_squared_error(yte, pred_test_ridge))

# --- MLP (pipeline tối ưu)
mlp = MLPRegressor(hidden_layer_sizes=(128,64),
                   solver="adam",
                   alpha=1e-4,
                   batch_size=256,
                   learning_rate_init=1e-3,
                   max_iter=400,
                   random_state=42,
                   early_stopping=True,
                   validation_fraction=0.15,
                   n_iter_no_change=12)
mlp_pipe = Pipeline([
    ("pre", pre_ridge),
    ("svd", TruncatedSVD(n_components=128, random_state=42)),
    ("sc2", StandardScaler()),
    ("est", MultiOutputRegressor(mlp))
])
mlp_pipe.fit(Xtr, ytr)

pred_train_mlp = mlp_pipe.predict(Xtr)
pred_test_mlp  = mlp_pipe.predict(Xte)
mlp_train_rmse = np.sqrt(mean_squared_error(ytr, pred_train_mlp))
mlp_test_rmse  = np.sqrt(mean_squared_error(yte, pred_test_mlp))

# --- Tổng hợp kết quả
df_overfit = pd.DataFrame({
    "Model": ["RidgeCV (Optimized)", "MLP (Optimized)"],
    "Train RMSE": [ridge_train_rmse, mlp_train_rmse],
    "Test RMSE": [ridge_test_rmse, mlp_test_rmse],
    "Overfit Gap (ΔRMSE)": [ridge_test_rmse - ridge_train_rmse,
                            mlp_test_rmse - mlp_train_rmse]
})
display(df_overfit)

# --- Biểu đồ trực quan
x = np.arange(len(df_overfit))
plt.figure(figsize=(7,4))
plt.bar(x-0.15, df_overfit["Train RMSE"], width=0.3, label="Train")
plt.bar(x+0.15, df_overfit["Test RMSE"], width=0.3, label="Test")
plt.xticks(x, df_overfit["Model"])
plt.ylabel("RMSE")
plt.title("So sánh Overfitting giữa RidgeCV và MLP (Pipeline tối ưu, 30k mẫu)")
plt.legend()
plt.show()

gc.collect();


Hiệu chỉnh để giảm overfit

In [ ]:
# ==============================
# Hiệu chỉnh Regularization cho MLP (Giảm Overfit + Fix OHE lỗi float/str)
# ==============================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, gc, warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

# ---- Dữ liệu mẫu
H = 6
D = dfs[H].copy()
D = add_dynamic_features(D)
D = D.sample(30000, random_state=42).reset_index(drop=True)

# ---- Xử lý categorical an toàn (tránh lỗi float/str)
def sanitize_categoricals(df, cat_cols):
    df = df.copy()
    for c in cat_cols:
        df[c] = df[c].astype(str)
        df[c] = df[c].replace(["nan", "None", "NaN", "NaT", "NULL"], "__MISSING__")
        df[c] = df[c].fillna("__MISSING__")
    return df

cat_cols = D.select_dtypes("object").columns.tolist()
num_cols = [c for c in D.select_dtypes("number").columns if c not in ("lat_tgt", "lon_tgt")]

if cat_cols:
    D = sanitize_categoricals(D, cat_cols)

# ---- Chia dữ liệu train/test
y_all = D[["lat_tgt", "lon_tgt"]].values
groups = D["storm_id"].astype(str).values
gss = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
tr_idx, te_idx = next(gss.split(D, y_all, groups))
Xtr, Xte = D.iloc[tr_idx], D.iloc[te_idx]
ytr, yte = y_all[tr_idx], y_all[te_idx]

# ---- Tạo OneHotEncoder an toàn (sparse cho RAM thấp)
def make_ohe_sparse():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True,
                             min_frequency=0.005, max_categories=64)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

# ---- Pipeline xử lý
num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler())
])

pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", make_ohe_sparse(), cat_cols)
], remainder="drop", sparse_threshold=1.0)

# ---- MLP Regularized (hiệu chỉnh giảm overfit)
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(64, 32),    # giảm số neuron
    solver="adam",
    alpha=1e-3,                     # regularization mạnh hơn
    batch_size=256,
    learning_rate_init=5e-4,        # học chậm hơn để ổn định
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=15,
    random_state=42
)

mlp_pipe_reg = Pipeline([
    ("pre", pre),
    ("svd", TruncatedSVD(n_components=128, random_state=42)),
    ("sc2", StandardScaler()),
    ("est", MultiOutputRegressor(mlp_reg))
])

# ---- Train và so sánh kết quả
mlp_pipe_reg.fit(Xtr, ytr)
pred_train = mlp_pipe_reg.predict(Xtr)
pred_test = mlp_pipe_reg.predict(Xte)

rmse_train = np.sqrt(mean_squared_error(ytr, pred_train))
rmse_test  = np.sqrt(mean_squared_error(yte, pred_test))
gap = rmse_test - rmse_train

print("✅ Hiệu chỉnh Regularization cho MLP (Reduced Overfit)")
print(f"Train RMSE: {rmse_train:.4f}")
print(f"Test RMSE:  {rmse_test:.4f}")
print(f"ΔRMSE (Overfit gap): {gap:.4f}")

# ---- Biểu đồ minh họa cải thiện
plt.figure(figsize=(6,4))
plt.bar(["Train", "Test"], [rmse_train, rmse_test], color=["skyblue", "salmon"])
plt.title("Hiệu quả sau khi Regularization MLP (Giảm Overfit)")
plt.ylabel("RMSE")
plt.show()

# Dọn bộ nhớ
gc.collect()


Trực quan hóa và đánh giá tương quan giữa phần dư (sai lệch dự đoán – thực tế) và bản thân đầu vào.

In [ ]:
# ==============================
# Trực quan hóa phần dư (Residual Analysis) — Bản ổn định nhất
# ==============================
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# --- Dữ liệu
H = 6
D = dfs[H].copy()
D = add_dynamic_features(D)
D = D.sample(30000, random_state=42).reset_index(drop=True)

# --- Làm sạch toàn bộ dữ liệu
cat_cols = D.select_dtypes("object").columns.tolist()
for c in cat_cols:
    D[c] = D[c].astype(str).fillna("__MISSING__")

num_cols = [c for c in D.select_dtypes("number").columns if c not in ("lat_tgt", "lon_tgt")]
y_all = D[["lat_tgt", "lon_tgt"]].values
groups = D["storm_id"].astype(str).values

# --- Chia nhóm train/test
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
tr_idx, te_idx = next(gss.split(D, y_all, groups))
Xte, yte = D.iloc[te_idx], y_all[te_idx]

# --- Các model đã huấn luyện
models = {
    "RidgeCV": ridge_pipe,
    "MLP (Regularized)": mlp_pipe_reg
}

def sanitize_for_predict(df, model):
    """Chuyển tất cả cột category sang string để tránh lỗi isnan."""
    df = df.copy()
    # Lấy danh sách cột mà pipeline đã học
    known_features = model.named_steps["pre"].transformers_[1][2]  # cat_cols trong preprocessor
    for c in known_features:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna("__MISSING__")
    return df

# --- Hàm vẽ phần dư
def plot_residuals(y_true, y_pred, name):
    residuals = y_true - y_pred
    res_mag = np.sqrt((residuals**2).sum(axis=1))
    rmse = np.sqrt(np.mean(res_mag**2))
    std = np.std(res_mag)

    print(f"\n📊 {name}: RMSE={rmse:.4f}, Std={std:.4f}")

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(res_mag, bins=40, kde=True, color="salmon")
    plt.title(f"Phân phối phần dư — {name}")
    plt.xlabel("Độ lớn phần dư")
    plt.ylabel("Tần suất")

    plt.subplot(1,2,2)
    sns.scatterplot(x=y_pred[:,0], y=res_mag, alpha=0.4, s=10, color="royalblue")
    plt.title(f"Phần dư theo vĩ độ dự đoán — {name}")
    plt.xlabel("Vĩ độ dự đoán (lat_pred)")
    plt.ylabel("Độ lớn phần dư")
    plt.tight_layout()
    plt.show()

# --- Thực thi
for name, model in models.items():
    Xsafe = sanitize_for_predict(Xte, model)
    y_pred = model.predict(Xsafe)
    plot_residuals(yte, y_pred, name)


In [ ]:
# ==============================
# Đánh giá mức độ phù hợp mô hình — So sánh RidgeCV vs MLP Regularized
# ==============================
import numpy as np
import matplotlib.pyplot as plt

def model_residual_stats(name, model, X, y):
    """Tính RMSE & std phần dư cho từng mô hình"""
    y_pred = model.predict(X)
    residuals = y - y_pred
    res_mag = np.sqrt((residuals**2).sum(axis=1))
    rmse = np.sqrt(np.mean(res_mag**2))
    std = np.std(res_mag)
    return {"Model": name, "RMSE": rmse, "Std": std}

# --- làm sạch dữ liệu đầu vào
def sanitize_for_predict(df, model):
    df = df.copy()
    cat_cols = model.named_steps["pre"].transformers_[1][2]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna("__MISSING__")
    return df

Xridge = sanitize_for_predict(Xte, ridge_pipe)
Xmlp   = sanitize_for_predict(Xte, mlp_pipe_reg)

ridge_stats = model_residual_stats("RidgeCV", ridge_pipe, Xridge, yte)
mlp_stats   = model_residual_stats("MLP (Regularized)", mlp_pipe_reg, Xmlp, yte)

df_eval = pd.DataFrame([ridge_stats, mlp_stats])
display(df_eval)

# --- Biểu đồ so sánh
plt.figure(figsize=(7,4))
x = np.arange(len(df_eval))
plt.bar(x - 0.2, df_eval["RMSE"], width=0.4, label="RMSE (Trung bình phần dư)")
plt.bar(x + 0.2, df_eval["Std"], width=0.4, label="Độ lệch chuẩn phần dư")
plt.xticks(x, df_eval["Model"])
plt.ylabel("Giá trị")
plt.title("So sánh độ phù hợp giữa các mô hình (Residual RMSE & Std)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# --- Nhận xét gợi ý in ra console
better = df_eval.loc[df_eval["RMSE"].idxmin(), "Model"]
print(f"✅ Mô hình có sai số phần dư thấp và ổn định nhất: **{better}**")
if better == "RidgeCV":
    print("➡ RidgeCV có độ ổn định cao và không overfit, rất phù hợp với dữ liệu có quan hệ tuyến tính mạnh.")
else:
    print("➡ MLP Regularized học tốt quan hệ phi tuyến, phù hợp khi muốn mở rộng phạm vi dự đoán phức tạp hơn.")


#1.3. Phân cụm dữ liệu:

In [ ]:
# ==============================
# Cell 1.3 — Clustering (K-Means & GMM) on INPUT features
# ==============================

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1) Lấy đặc trưng đầu vào (bỏ toàn bộ output & derived output để tránh leakage)
drop_cols = [
    "lat_tgt", "lon_tgt",
    "lat_baseline", "lon_baseline",
    "lat_dev", "lon_dev",
    "lat_qlabel", "lon_qlabel",
    "out_km_label", "out_gmm_label"
]

X_cluster = D_outlabels.select_dtypes(include=[np.number]) \
                       .drop(columns=[c for c in drop_cols if c in D_outlabels.columns],
                             errors="ignore")

# 2) Chuẩn hóa (dùng lại preprocessing nếu đã có, nếu chưa thì làm tại đây)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imp = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imp = imp.fit_transform(X_cluster)
X_scaled = scaler.fit_transform(X_imp)

print("Input shape for clustering:", X_scaled.shape)

# 3) Giảm chiều PCA để phân cụm + trực quan
pca = PCA(n_components=6, random_state=RND)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance (6 PCs):",
      np.round(pca.explained_variance_ratio_ * 100, 2))
print("Cumulative:",
      np.round(pca.explained_variance_ratio_.cumsum()[-1] * 100, 2), "%")

# 4) Phân cụm với K-Means (k = 4 theo thiết kế bài toán)
k = 4
kmeans = KMeans(n_clusters=k, random_state=RND, n_init="auto")
km_labels = kmeans.fit_predict(X_pca)

# 5) Phân cụm với GMM (k = 4)
gmm = GaussianMixture(n_components=k, random_state=RND)
gmm_labels = gmm.fit_predict(X_pca)

# 6) Đánh giá định lượng chất lượng phân cụm
print("\n=== Clustering Quality Metrics ===")
print("KMeans  - Silhouette:", silhouette_score(X_pca, km_labels))
print("KMeans  - Davies–Bouldin:", davies_bouldin_score(X_pca, km_labels))

print("GMM     - Silhouette:", silhouette_score(X_pca, gmm_labels))
print("GMM     - Davies–Bouldin:", davies_bouldin_score(X_pca, gmm_labels))

# 7) Trực quan hóa trên PCA(2)
pca2 = PCA(n_components=2, random_state=RND)
X_pca2 = pca2.fit_transform(X_scaled)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
sns.scatterplot(
    x=X_pca2[:,0], y=X_pca2[:,1],
    hue=km_labels, palette="tab10",
    s=10, legend="full"
)
plt.title("K-Means Clustering on PCA(2)")
plt.xlabel("PC1"); plt.ylabel("PC2")

plt.subplot(1,2,2)
sns.scatterplot(
    x=X_pca2[:,0], y=X_pca2[:,1],
    hue=gmm_labels, palette="tab10",
    s=10, legend="full"
)
plt.title("GMM Clustering on PCA(2)")
plt.xlabel("PC1"); plt.ylabel("PC2")

plt.tight_layout()
plt.show()

# 8) Lưu nhãn phân cụm để dùng cho các bước sau (phân tích output / classification)
D_outlabels["cluster_kmeans"] = km_labels
D_outlabels["cluster_gmm"] = gmm_labels

print("\nCluster distribution (KMeans):")
print(pd.Series(km_labels).value_counts().sort_index())

print("\nCluster distribution (GMM):")
print(pd.Series(gmm_labels).value_counts().sort_index())


## Nhận xét về mối quan hệ giữa các mẫu dữ liệu và các đầu ra trong các cụm

### 1. Mối quan hệ giữa các mẫu dữ liệu đầu vào trong các cụm

Kết quả phân cụm được đánh giá dựa trên các độ đo định lượng gồm **Silhouette score** và **Davies–Bouldin index**, kết hợp với trực quan hóa trên không gian **PCA(2)**.

- **Silhouette score**:
  - KMeans: **0.2017**
  - GMM: **0.1266**

Giá trị Silhouette của cả hai mô hình đều **lớn hơn 0**, cho thấy các mẫu trong cùng một cụm có xu hướng gần nhau hơn so với các mẫu thuộc cụm khác. Tuy nhiên, giá trị này còn thấp (< 0.25), phản ánh mức độ **chồng lấn tương đối lớn giữa các cụm**. So sánh hai phương pháp, KMeans thể hiện khả năng phân tách cụm tốt hơn GMM.

- **Davies–Bouldin index**:
  - KMeans: **1.5154**
  - GMM: **1.8584**

Chỉ số Davies–Bouldin càng nhỏ thì chất lượng phân cụm càng cao. Kết quả cho thấy KMeans tạo ra các cụm **gọn hơn và ít chồng lấn hơn** so với GMM.

Quan sát trực quan trên PCA(2) cho thấy các cụm có cấu trúc kéo dài theo trục chính (PC1), phản ánh sự tương quan mạnh giữa các biến đầu vào. Các cụm không tách rời hoàn toàn mà có vùng giao nhau, phù hợp với các giá trị Silhouette và Davies–Bouldin đã nêu.

---

### 2. Quan hệ giữa các đầu ra tương ứng trong các cụm

Phân bố số lượng mẫu trong từng cụm cho thấy các cụm không cân bằng hoàn toàn nhưng không có cụm quá nhỏ:

- **KMeans**: các cụm có kích thước từ khoảng **8,000 đến 53,000 mẫu**
- **GMM**: các cụm có kích thước từ khoảng **16,000 đến 40,000 mẫu**

Điều này cho thấy mỗi cụm đại diện cho một nhóm hành vi đầu ra khác nhau, thay vì chỉ là nhiễu hoặc ngoại lệ.

Trong không gian PCA, các cụm có xu hướng chiếm các vùng khác nhau dọc theo PC1 và PC2, cho thấy các **đầu ra trong cùng một cụm có quan hệ gần nhau**, còn giữa các cụm tồn tại sự khác biệt có hệ thống. Tuy nhiên, do mức độ chồng lấn còn đáng kể, ranh giới giữa các nhóm đầu ra mang tính **mờ (fuzzy)** hơn là tách biệt hoàn toàn.

---

### 3. Kết luận

Dựa trên các độ đo định lượng và trực quan hóa:
- KMeans cho chất lượng phân cụm tốt hơn GMM trong bài toán này.
- Các cụm phản ánh được cấu trúc tổng thể của dữ liệu, nhưng mức độ tách biệt chưa cao.
- Quan hệ giữa các đầu ra trong từng cụm là có ý nghĩa, song vẫn tồn tại sự giao thoa, cho thấy dữ liệu có tính liên tục cao hơn là phân nhóm rời rạc hoàn toàn.


#c) Bài toán phân loại

In [ ]:
# ==============================
# Cell A: Baseline + Deviation + Labeling (Semantic)
# ==============================
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

H = 6
D = dfs[H].copy().reset_index(drop=True)

# 1) Baseline tuyến tính
D["lat_baseline"] = D["lat_lag1"] + (D["lat_lag1"] - D["lat_lag2"])
D["lon_baseline"] = D["lon_lag1"] + (D["lon_lag1"] - D["lon_lag2"])

D = D.dropna(subset=["lat_baseline", "lon_baseline"]).reset_index(drop=True)

# 2) Deviation
D["lat_dev"] = D["lat_tgt"] - D["lat_baseline"]
D["lon_dev"] = D["lon_tgt"] - D["lon_baseline"]

# 3) Quantile labels có NGỮ NGHĨA
def make_semantic_quantile_labels(series):
    q = pd.qcut(series, q=4)
    return pd.cut(
        series,
        bins=q.cat.categories,
        labels=["Neg_Large", "Neg_Small", "Pos_Small", "Pos_Large"],
        include_lowest=True
    )

D["lat_label"] = make_semantic_quantile_labels(D["lat_dev"])
D["lon_label"] = make_semantic_quantile_labels(D["lon_dev"])

# 4) Output clustering labels
out_features = D[["lat_dev", "lon_dev"]].to_numpy()

km = KMeans(n_clusters=4, random_state=RND, n_init="auto")
D["out_km_label"] = km.fit_predict(out_features).astype(str)

gmm = GaussianMixture(n_components=4, random_state=RND)
D["out_gmm_label"] = gmm.fit_predict(out_features).astype(str)

print("Lat label distribution:")
print(D["lat_label"].value_counts())

D_outlabels = D.copy()


In [ ]:
# ==============================
# Cell — Quick check D after labeling
# ==============================

cols_check = [
    "storm_id", "time",
    "lat_lag1", "lat_lag2",
    "lat_baseline", "lat_tgt", "lat_dev", "lat_label",
    "lon_baseline", "lon_tgt", "lon_dev", "lon_label",
    "out_km_label", "out_gmm_label"
]

print("=== First 5 rows of D ===")
display(D[cols_check].head())


In [ ]:
print(df.columns.tolist())


In [ ]:
import numpy as np

D["dev_mag"] = np.sqrt(
    D["lat_dev"]**2 + D["lon_dev"]**2
)

# kiểm tra nhanh
print(D["dev_mag"].describe())


In [ ]:
class_names = [
    "Small_Error",
    "Medium_Small",
    "Medium_Large",
    "Large_Error"
]

D["cls_label"] = pd.qcut(
    D["dev_mag"],
    q=4,
    labels=class_names
)

# Kiểm tra
print("=== Final classification label distribution ===")
print(D["cls_label"].value_counts())

display(D[["dev_mag", "cls_label"]].head())


Trong bước này, một baseline tuyến tính được xây dựng từ các giá trị trễ nhằm mô
phỏng xu hướng chuyển động ngắn hạn của đối tượng, chỉ sử dụng thông tin quá khứ
để tránh rò rỉ dữ liệu. Độ lệch (deviation) giữa giá trị mục tiêu thực tế và
baseline được xem là đại diện cho sai số dự báo.

Để chuyển bài toán từ hồi quy sang phân loại, độ lớn sai lệch được chia thành
bốn khoảng theo phân vị (quantile) sao cho số lượng mẫu trong mỗi lớp xấp xỉ
nhau. Các nhãn thu được mang ý nghĩa ngữ nghĩa về mức độ sai số, phục vụ cho
các thí nghiệm phân loại và so sánh mô hình ở các bước tiếp theo.

In [ ]:
# ==============================
# Cell B: PCA + Clustering on Input Features
# ==============================
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

drop_cols = [
    "lat_tgt","lon_tgt",
    "lat_baseline","lon_baseline",
    "lat_dev","lon_dev",
    "lat_label","lon_label",
    "out_km_label","out_gmm_label"
]

X = D_outlabels.select_dtypes(include=[np.number]) \
               .drop(columns=[c for c in drop_cols if c in D_outlabels.columns],
                     errors="ignore")

X = SimpleImputer(strategy="median").fit_transform(X)
X = StandardScaler().fit_transform(X)

pca6 = PCA(n_components=6, random_state=RND)
X_pca6 = pca6.fit_transform(X)

print("Explained variance (6 PCs):",
      np.round(pca6.explained_variance_ratio_ * 100, 2))
print("Cumulative:",
      np.round(pca6.explained_variance_ratio_.cumsum()[-1] * 100, 2), "%")

sil_scores = {}
for k in [2, 3, 4]:
    lbl = KMeans(k, random_state=RND, n_init="auto").fit_predict(X_pca6)
    sil_scores[k] = silhouette_score(X_pca6, lbl)
    print(f"k={k}, silhouette={sil_scores[k]:.4f}")

best_k = max(sil_scores, key=sil_scores.get)
print("Best k:", best_k)

X_pca2 = PCA(n_components=2, random_state=RND).fit_transform(X)
labels = KMeans(best_k, random_state=RND, n_init="auto").fit_predict(X_pca2)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1],
                hue=labels, palette="tab10", s=10)
plt.title(f"KMeans on PCA(2), k={best_k}")
plt.show()


In [ ]:
# ==============================
# Cell C: Prepare X, y for Classification
# ==============================
from sklearn.preprocessing import LabelEncoder

label_choice = "quantile"  # "quantile" hoặc "out_cluster"

D = D_outlabels.copy()

if label_choice == "quantile":
    y = D["lat_label"]
else:
    y = D["out_km_label"]

X = D.select_dtypes(include=[np.number]) \
     .drop(columns=[c for c in drop_cols if c in D.columns],
           errors="ignore")

mask = ~y.isna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)
groups = D.loc[mask, "storm_id"].reset_index(drop=True)

print("Label distribution:")
print(y.value_counts())


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
import seaborn as sns
import matplotlib.pyplot as plt

splits = {
    "4:1": 0.20,
    "7:3": 0.30,
    "6:4": 0.40
}

for split_name, test_size in splits.items():
    print("\n" + "="*50)
    print(f" Train : Test = {split_name}")
    print("="*50)

    # ===== FIXED SPLIT: group theo storm_id =====
    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=RND
    )
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # ===========================================

    # encode label
    y_train_str = y_train.astype(str)
    y_test_str  = y_test.astype(str)
    le = LabelEncoder()
    ytr = le.fit_transform(y_train_str)
    yte = le.transform(y_test_str)
    class_names = le.classes_.tolist()

    # train model
    nb = GaussianNB()
    rf = RandomForestClassifier(
        n_estimators=150,
        random_state=RND,
        n_jobs=-1
    )
    nb.fit(Xtr, ytr)
    rf.fit(Xtr, ytr)

    # evaluate
    for name, model in [("Naive Bayes", nb), ("Random Forest", rf)]:
        pred = model.predict(Xte)
        print(f"\n{name}")
        print("F1-macro:", f1_score(yte, pred, average="macro"))
        print(classification_report(
            yte,
            pred,
            target_names=class_names,
            digits=4
        ))

    # confusion matrices
    cm_nb = confusion_matrix(yte, nb.predict(Xte))
    cm_rf = confusion_matrix(yte, rf.predict(Xte))

    plt.figure(figsize=(12,5))

    # Naive Bayes
    plt.subplot(1, 2, 1)
    sns.heatmap(
        cm_nb,
        annot=True,
        fmt="d",
        cmap="Oranges",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f"Naive Bayes ({split_name})")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    # Random Forest
    plt.subplot(1, 2, 2)
    sns.heatmap(
        cm_rf,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f"Random Forest ({split_name})")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    plt.tight_layout()
    plt.show()


## Đánh giá và so sánh kết quả thực nghiệm

### 1. So sánh giữa các mô hình

- **Random Forest** cho kết quả vượt trội rõ rệt so với **Naive Bayes** trong mọi tỷ lệ Train:Test.
  - F1-macro của Random Forest ổn định quanh **0.69**
  - F1-macro của Naive Bayes chỉ đạt khoảng **0.56–0.57**

Nguyên nhân chính là Naive Bayes giả định **các đặc trưng độc lập có điều kiện**, trong khi dữ liệu bài toán (độ lệch quỹ đạo bão) có **tương quan mạnh giữa các biến không gian – thời gian**. Random Forest khai thác tốt các quan hệ phi tuyến và tương tác giữa đặc trưng, nên đạt hiệu năng cao hơn.

---

### 2. So sánh giữa các tỷ lệ Train : Test

- Kết quả của cả hai mô hình **gần như không thay đổi** khi thay đổi tỷ lệ 4:1, 7:3 và 6:4.
- Random Forest giữ F1-macro rất ổn định (≈ 0.69), cho thấy mô hình:
  - Không bị overfitting
  - Có khả năng tổng quát hóa tốt theo từng cơn bão (group theo `storm_id`)

Điều này cho thấy **kích thước dữ liệu huấn luyện đã đủ lớn**, việc tăng thêm dữ liệu train không mang lại cải thiện đáng kể.

---

### 3. Phân tích theo lớp nhãn

- Các lớp biên lớn như **(-3.201, 0.1]** và **(0.8, 7.4]** có F1-score cao hơn.
- Các lớp trung gian **(0.1, 0.4]** và **(0.4, 0.8]** khó phân loại hơn do:
  - Biên giá trị chồng lấn
  - Độ lệch nhỏ, dễ bị nhiễu đo đạc

---

### 4. Kết luận

- Random Forest là lựa chọn phù hợp hơn cho bài toán phân loại độ lệch quỹ đạo bão.
- Việc thay đổi tỷ lệ Train:Test không ảnh hưởng đáng kể đến kết quả.
- Khó khăn chính nằm ở việc phân biệt các mức độ lệch nhỏ, phản ánh tính liên tục của hiện tượng vật lý.


In [ ]:
# ==============================
# Cell F: Summary & Quick Observations
# ==============================

results = {
    "4:1": {"NB": 0.5741, "RF": 0.7109},
    "7:3": {"NB": 0.5738, "RF": 0.7086},
    "6:4": {"NB": 0.5745, "RF": 0.7041}
}

print("Quick Observations:")
print("-" * 50)

for split, metrics in results.items():
    print(f"Train:Test = {split}")
    print(f"  - Naive Bayes F1-macro: {metrics['NB']:.4f}")
    print(f"  - Random Forest F1-macro: {metrics['RF']:.4f}")

    # Nhận xét ngắn
    if metrics['RF'] > metrics['NB']:
        print("    => RF consistently outperforms NB across splits.")
    print()

print("Overall:")
print(" - F1-macro của NB ổn định ~0.574, không thay đổi nhiều theo split.")
print(" - RF cho F1-macro cao hơn NB (~0.705–0.711), hơi giảm khi test size tăng.")
print(" - Split nhỏ hơn (4:1) hơi tốt hơn cho RF, nhưng sự khác biệt không lớn.")


Random Forest với dữ liệu giảm chiều

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# 1/3 dims
k = max(1, Xtr.shape[1] // 3)

# chuẩn hóa trước PCA
scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)

pca = PCA(n_components=k, random_state=RND)
Xtr_p = pca.fit_transform(Xtr_scaled)
Xte_p = pca.transform(Xte_scaled)

print("PCA cumulative variance:",
      np.round(pca.explained_variance_ratio_.cumsum()[-1]*100, 2), "%")

# train Random Forest trên PCA
rf_p = RandomForestClassifier(n_estimators=150, random_state=RND, n_jobs=-1)
rf_p.fit(Xtr_p, ytr)

pred_p = rf_p.predict(Xte_p)
print("RF PCA F1-macro:", f1_score(yte, pred_p, average="macro"))


Naive Bayes với dữ liệu giảm chiều

In [ ]:
from sklearn.naive_bayes import GaussianNB

# ==============================
# Naive Bayes trên dữ liệu PCA
# ==============================

nb_p = GaussianNB()
nb_p.fit(Xtr_p, ytr)

pred_nb_p = nb_p.predict(Xte_p)

print("NB PCA F1-macro:", f1_score(yte, pred_nb_p, average="macro"))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== Data (UPDATED after GroupSplit) =====
splits = ["4:1", "7:3", "6:4"]

# Without PCA
nb_no_pca = [0.5624, 0.5700, 0.5692]
rf_no_pca = [0.6870, 0.6903, 0.6900]

# With PCA (~92.81% variance retained)
nb_pca = [0.5292] * 3
rf_pca = [0.6131] * 3

x = np.arange(len(splits))
width = 0.35

plt.figure(figsize=(12,4))

# ===== Plot 1: No PCA =====
plt.subplot(1, 2, 1)
plt.bar(x - width/2, nb_no_pca, width, label="Naive Bayes")
plt.bar(x + width/2, rf_no_pca, width, label="Random Forest")
plt.xticks(x, splits)
plt.ylim(0.5, 0.75)
plt.title("Without PCA (Group Split)")
plt.ylabel("F1-macro")
plt.xlabel("Train : Test split")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.6)

# ===== Plot 2: With PCA =====
plt.subplot(1, 2, 2)
plt.bar(x - width/2, nb_pca, width, label="Naive Bayes")
plt.bar(x + width/2, rf_pca, width, label="Random Forest")
plt.xticks(x, splits)
plt.ylim(0.5, 0.75)
plt.title("With PCA (~92.8% variance retained)")
plt.xlabel("Train : Test split")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.suptitle("F1-macro Comparison of Models (Group-aware Split)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================
# Cell G: Label Comparison
# ==============================
print(pd.crosstab(D_outlabels["lat_label"],
                  D_outlabels["out_km_label"]))


In [ ]:
# ==============================
# Cell H: Text summary — true vs predicted label distribution (RF)
# ==============================
import pandas as pd

y_true_lbl = le.inverse_transform(yte)
y_pred_lbl = le.inverse_transform(rf.predict(Xte))

print("=== TRUE label distribution ===")
print(pd.Series(y_true_lbl).value_counts().sort_index())

print("\n=== PREDICTED label distribution (RF) ===")
print(pd.Series(y_pred_lbl).value_counts().sort_index())


In [ ]:
# ==============================
# Cell I: Most frequent misclassifications (RF)
# ==============================
df_err = pd.DataFrame({
    "true": y_true_lbl,
    "pred": y_pred_lbl
})

df_err = df_err[df_err["true"] != df_err["pred"]]

print("=== Top 10 confusions ===")
print(
    df_err.value_counts()
          .rename("count")
          .reset_index()
          .sort_values("count", ascending=False)
          .head(10)
)


In [ ]:
# ==============================
# Cell J: Per-class accuracy (RF)
# ==============================
per_class_acc = (
    df_err.groupby("true").size()
    .rsub(pd.Series(y_true_lbl).value_counts())
    / pd.Series(y_true_lbl).value_counts()
)

print("=== Per-class accuracy ===")
print(per_class_acc.sort_index())


In [ ]:
# ==============================
# Cell K: Model comparison (TEXT)
# ==============================
from sklearn.metrics import f1_score, accuracy_score

models = {
    "Naive Bayes": nb,
    "Random Forest": rf
}

rows = []
for name, model in models.items():
    pred = model.predict(Xte)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(yte, pred),
        "f1_macro": f1_score(yte, pred, average="macro")
    })

print(pd.DataFrame(rows))


XGBoost (Thử nghiệm)

In [ ]:
# ==============================
# Cell F: XGBoost Classification
# ==============================
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Encode labels (numeric for model)
le = LabelEncoder()
ytr = le.fit_transform(y_train)
yte = le.transform(y_test)

# Convert class names to string for reporting
class_names = [str(c) for c in le.classes_]

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(class_names),
    eval_metric="mlogloss",
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=RND,
    n_jobs=-1
)

xgb.fit(Xtr, ytr)

pred = xgb.predict(Xte)

print("\nXGBoost")
print("F1-macro:", f1_score(yte, pred, average="macro"))
print(classification_report(yte, pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(yte, pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("Confusion Matrix — XGBoost")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ==============================
# Cell F1: Train vs Test performance (Overfitting check)
# ==============================
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

models = {
    "Naive Bayes": nb,
    "Random Forest": rf,
    "XGBoost": xgb
}

train_scores = []
test_scores = []

for name, model in models.items():
    train_scores.append(
        f1_score(ytr, model.predict(Xtr), average="macro")
    )
    test_scores.append(
        f1_score(yte, model.predict(Xte), average="macro")
    )

plt.figure(figsize=(8,5))
plt.bar(models.keys(), train_scores, alpha=0.7, label="Train")
plt.bar(models.keys(), test_scores, alpha=0.7, label="Test")
plt.ylabel("F1-macro")
plt.title("Train vs Test Performance (Overfitting Analysis)")
plt.legend()
plt.ylim(0.5, 0.8)
plt.show()


In [ ]:
# ==============================
# Cell F2: Accuracy vs F1-macro
# ==============================
import pandas as pd

summary = pd.DataFrame({
    "Model": ["Naive Bayes", "Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(yte, nb.predict(Xte)),
        accuracy_score(yte, rf.predict(Xte)),
        accuracy_score(yte, xgb.predict(Xte))
    ],
    "F1_macro": test_scores
})

summary.set_index("Model").plot(
    kind="bar",
    figsize=(8,5),
    rot=0
)

plt.title("Model Effectiveness Comparison")
plt.ylabel("Score")
plt.ylim(0.5, 0.8)
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# ==============================
# Cell F3: Prediction confidence distribution (XGBoost)
# ==============================
proba = xgb.predict_proba(Xte)
max_proba = proba.max(axis=1)

plt.figure(figsize=(7,4))
plt.hist(max_proba, bins=30, edgecolor="black")
plt.xlabel("Max predicted probability")
plt.ylabel("Count")
plt.title("Prediction Confidence Distribution (XGBoost)")
plt.show()


In [ ]:
# ==============================
# Cell Z: Data Leakage Check (Sanity Check)
# ==============================

print("=== DATA LEAKAGE SANITY CHECK ===\n")

# Các cột tuyệt đối KHÔNG được xuất hiện trong feature X
forbidden_cols = [
    "lat_tgt", "lon_tgt",
    "lat_dev", "lon_dev",
    "dev_mag",
    "lat_label", "lon_label", "cls_label",
    "out_km_label", "out_gmm_label"
]

print("Forbidden columns present in X:")
leak_found = False
for col in forbidden_cols:
    present = col in X.columns
    print(f"{col:15s}: {present}")
    if present:
        leak_found = True

print("\n=== RESULT ===")
if leak_found:
    print("Potential DATA LEAKAGE detected! Check feature construction.")
else:
    print("No data leakage detected. Feature set is clean.")

print("\nFeature count:", X.shape[1])
print("Label used:", y.name if hasattr(y, "name") else "encoded label")


In [ ]:
# ==============================
# Cell: List Numerical Features Used in Model
# ==============================
import numpy as np

num_features = X.columns.tolist()

print(f"Total numerical features used: {len(num_features)}\n")
print("List of numerical features:\n")
for i, col in enumerate(num_features, 1):
    print(f"{i:02d}. {col}")
